In [ ]:
import argparse
import glob
import os

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
mod_label = ['Slab', 'Urb', 'Urb_Veg', 'CLM5U']

In [ ]:
label_sites = [
    'AU-Preston', 'AU-SurreyHills', 'CA-Sunset', 'FI-Kumpula', 'FI-Torni', 'FR-Capitole',
    'GR-HECKOR', 'JP-Yoyogi', 'KR-Jungnang', 'KR-Ochang', 'MX-Escandon', 'NL-Amsterdam',
    'PL-Lipowa', 'PL-Narutowicza', 'SG-TelokKurau06', 'UK-KingsCollege', 'UK-Swindon', 'US-Baltimore',
    'US-Minneapolis1', 'US-Minneapolis2', 'US-WestPhoenix'
]

In [ ]:
PATH_SLAB = '../../model_output/slab/'
PATH_URB = '../../model_output/urb/'
PATH_VEG = '../../model_output/veg/'
PATH_CLM5 = '../../model_output/clm5'
PATH_OBS = '../../obs/'

In [ ]:
def calculate_metrics(observed, predicted):
    valid_indices = ~np.isnan(observed) & ~np.isnan(predicted)
    observed = observed[valid_indices]
    predicted = predicted[valid_indices]

    correlation_coefficient_, _ = np.corrcoef(observed, predicted)
    correlation_coefficient = correlation_coefficient_[1]
    rmse = np.sqrt(np.mean((predicted - observed) ** 2))
    mbe = np.mean(predicted - observed)
    std_ratio = np.std(predicted) / np.std(observed)

    print(correlation_coefficient, rmse, mbe, std_ratio)
    return correlation_coefficient, rmse, mbe, std_ratio

In [ ]:
def get_var_lists(var_type):
    if var_type == 'turbulent':
        var_mod = ['f_fsena', 'f_lfevpa', 'f_fgrnd']
        var_obs = ['Qh', 'Qle', 'Qg']
        row_titles = ('Sensible Heat', 'Latent Heat', 'Storage Heat')
        outname = 'taylor_diagram_turbulent'
    elif var_type == 'radiation':
        var_mod = ['f_sr', 'f_olrg', 'f_rnet']
        var_obs = ['SWup', 'LWup', 'Rnet']
        row_titles = ('Upward Shortwave', 'Upward Longwave', 'Net Radiation')
        outname = 'taylor_diagram_radiation'
    else:
        raise ValueError("var_type must be 'turbulent' or 'radiation'")
    return var_mod, var_obs, row_titles, outname

In [ ]:
def get_paths_for_model(model):
    if model == 'Slab':
        return PATH_SLAB, 'nc'
    if model == 'Urb':
        return PATH_URB, 'nc'
    if model == 'Urb_Veg':
        return PATH_VEG, 'nc'
    if model == 'CLM5U':
        return PATH_CLM5, 'csv'
    raise ValueError(model)

In [ ]:
def find_history_files(base_path, site):
    pattern = os.path.join(base_path, site, 'history', '*.nc')
    return sorted(glob.glob(pattern))

In [ ]:
def open_history_dataset(base_path, site):
    files = find_history_files(base_path, site)
    if not files:
        return None

    if len(files) == 1:
        ds = xr.open_dataset(files[0])
    else:
        datasets = [xr.open_dataset(path) for path in files]
        ds = xr.concat(datasets, dim='time', data_vars='minimal', coords='minimal', compat='override')
        ds = ds.sortby('time')
        ds.load()
        for dataset in datasets:
            dataset.close()

    if 'patch' in ds.dims:
        ds = ds.isel(patch=0, drop=True)
    return ds

In [ ]:
def open_reference_model_dataset(site):
    for model in ['Slab', 'Urb', 'Urb_Veg']:
        base_path, ftype = get_paths_for_model(model)
        if ftype != 'nc':
            continue

        ds = open_history_dataset(base_path, site)
        if ds is not None:
            return ds

    return None

In [ ]:
def build_clm_obs_alignment(clm_df, obs_ds, ref_ds):
    clm_times = pd.to_datetime(clm_df['time'])
    swdown = xr.where(ref_ds['f_xy_solarin'] == 0, 0, obs_ds['SWdown'][:-1])
    swup = xr.where(ref_ds['f_xy_solarin'] == 0, 0, obs_ds['SWup'][:-1])

    obs_frame = pd.DataFrame(
        {
            'time': pd.to_datetime(ref_ds['time'].values),
            'SWdown': swdown.values,
            'LWdown': obs_ds['LWdown'][:-1].values,
            'SWup': swup.values,
            'LWup': obs_ds['LWup'][:-1].values,
            'Qh': obs_ds['Qh'][:-1].values,
            'Qle': obs_ds['Qle'][:-1].values,
        }
    )
    return pd.DataFrame({'time': clm_times}).merge(obs_frame, on='time', how='left')

In [ ]:
def calculate_metrics_to_csv(var_type, out_csv):
    var_mod, var_obs, _, _ = get_var_lists(var_type)
    rows = []

    for site_i, site in enumerate(label_sites, start=1):
        obs_file = f'{PATH_OBS}/{site}_clean_observations_v1.nc'
        obs_ds = xr.open_dataset(obs_file)
        clm_ref_ds = open_reference_model_dataset(site)

        for model in mod_label:
            print(f'[Calculatiung] {var_type} | {site} of {model}')
            base_path, ftype = get_paths_for_model(model)

            if ftype == 'csv':
                df = pd.read_csv(f'{base_path}/{site}.csv')

                for obs_var in var_obs:
                    if (site == 'MX-Escandon') and (obs_var in ['LWup', 'Rnet', 'Qg']):
                        r, rmse, mbe, std_ratio = np.nan, np.nan, np.nan, np.nan
                    else:
                        if obs_var not in ['Rnet', 'Qg']:
                            if f'{obs_var}_obs' not in df.columns or f'{obs_var}_cesmlcz' not in df.columns:
                                r, rmse, mbe, std_ratio = np.nan, np.nan, np.nan, np.nan
                            else:
                                r, rmse, mbe, std_ratio = calculate_metrics(df[f'{obs_var}_obs'], df[f'{obs_var}_cesmlcz'])
                        elif obs_var == 'Rnet':
                            required_cols = ['SWup_cesmlcz', 'LWup_cesmlcz']
                            if clm_ref_ds is None or any(col not in df.columns for col in required_cols):
                                r, rmse, mbe, std_ratio = np.nan, np.nan, np.nan, np.nan
                            else:
                                aligned_obs = build_clm_obs_alignment(df, obs_ds, clm_ref_ds)
                                clm_swup = df['SWup_cesmlcz'].where(aligned_obs['SWdown'] != 0, 0)
                                obs_rnet = aligned_obs['SWdown'] + aligned_obs['LWdown'] - aligned_obs['SWup'] - aligned_obs['LWup']
                                mod_rnet = aligned_obs['SWdown'] + aligned_obs['LWdown'] - clm_swup - df['LWup_cesmlcz']
                                r, rmse, mbe, std_ratio = calculate_metrics(obs_rnet, mod_rnet)
                        else:
                            required_cols = ['SWup_cesmlcz', 'LWup_cesmlcz', 'Qh_cesmlcz', 'Qle_cesmlcz']
                            if clm_ref_ds is None or any(col not in df.columns for col in required_cols):
                                r, rmse, mbe, std_ratio = np.nan, np.nan, np.nan, np.nan
                            else:
                                aligned_obs = build_clm_obs_alignment(df, obs_ds, clm_ref_ds)
                                clm_swup = df['SWup_cesmlcz'].where(aligned_obs['SWdown'] != 0, 0)
                                obs_rnet = aligned_obs['SWdown'] + aligned_obs['LWdown'] - aligned_obs['SWup'] - aligned_obs['LWup']
                                mod_rnet = aligned_obs['SWdown'] + aligned_obs['LWdown'] - clm_swup - df['LWup_cesmlcz']
                                obs_qg = obs_rnet - aligned_obs['Qh'] - aligned_obs['Qle']
                                mod_qg = mod_rnet - df['Qh_cesmlcz'] - df['Qle_cesmlcz']
                                r, rmse, mbe, std_ratio = calculate_metrics(obs_qg, mod_qg)

                    rows.append(
                        dict(
                            var_type=var_type,
                            site=site,
                            site_i=site_i,
                            model=model,
                            var=obs_var,
                            r=r,
                            rmse=rmse,
                            mbe=mbe,
                            std_ratio=std_ratio,
                        )
                    )
            else:
                pred_ds = open_history_dataset(base_path, site)
                if pred_ds is None:
                    continue

                for obs_var, mod_var in zip(var_obs, var_mod):
                    if obs_var not in ['Rnet', 'Qg']:
                        obs_da = obs_ds[obs_var]
                        mod_da = pred_ds[mod_var]
                    else:
                        if obs_var == 'Qg':
                            mod_da = (
                                pred_ds['f_xy_solarin'] + pred_ds['f_xy_frl'] - pred_ds['f_sr'] - pred_ds['f_olrg']
                                - pred_ds['f_fsena'] - pred_ds['f_lfevpa']
                            )
                            swdown = xr.where(pred_ds['f_xy_solarin'] == 0, 0, obs_ds['SWdown'][:-1])
                            swup = xr.where(pred_ds['f_xy_solarin'] == 0, 0, obs_ds['SWup'][:-1])
                            lwdown = obs_ds['LWdown'][:-1]
                            lwup = obs_ds['LWup'][:-1]
                            obs_da = swdown + lwdown - swup - lwup - obs_ds['Qh'][:-1] - obs_ds['Qle'][:-1]
                        else:
                            mod_da = pred_ds['f_xy_solarin'] + pred_ds['f_xy_frl'] - pred_ds['f_sr'] - pred_ds['f_olrg']
                            swdown = xr.where(pred_ds['f_xy_solarin'] == 0, 0, obs_ds['SWdown'][:-1])
                            swup = xr.where(pred_ds['f_xy_solarin'] == 0, 0, obs_ds['SWup'][:-1])
                            lwdown = obs_ds['LWdown'][:-1]
                            lwup = obs_ds['LWup'][:-1]
                            obs_da = swdown + lwdown - swup - lwup

                    sim_start = max(mod_da['time'].min().values, obs_da['time'].min().values)
                    sim_end = min(mod_da['time'].max().values, obs_da['time'].max().values)

                    obs_f = obs_da.sel(time=slice(sim_start, sim_end)).dropna(dim='time')
                    mod_f = mod_da.sel(time=slice(sim_start, sim_end)).sel(time=obs_f['time'])

                    if (site == 'MX-Escandon') and (obs_var in ['LWup', 'Rnet', 'Qg']):
                        r, rmse, mbe, std_ratio = np.nan, np.nan, np.nan, np.nan
                    else:
                        r, rmse, mbe, std_ratio = calculate_metrics(obs_f.values, mod_f.values)

                    rows.append(
                        dict(
                            var_type=var_type,
                            site=site,
                            site_i=site_i,
                            model=model,
                            var=obs_var,
                            r=r,
                            rmse=rmse,
                            mbe=mbe,
                            std_ratio=std_ratio,
                        )
                    )

    df_out = pd.DataFrame(rows)
    if out_csv is not None:
        df_out.to_csv(out_csv, index=False)
        print(f'[Saved cache] {out_csv}')
    return df_out

In [ ]:
def calculate_all_metrics_to_csv(out_csv):
    frames = [
        calculate_metrics_to_csv('radiation', out_csv=None),
        calculate_metrics_to_csv('turbulent', out_csv=None),
    ]
    df_out = pd.concat(frames, ignore_index=True)
    df_out.to_csv(out_csv, index=False)
    print(f'[Saved cache] {out_csv}')
    return df_out

In [ ]:
def build_parser():
    parser = argparse.ArgumentParser(description='Calculate Taylor-diagram metrics and save to csv.')
    parser.add_argument('--metrics-csv', default='taylor_metrics.csv')
    return parser

In [ ]:
def main():
    args = build_parser().parse_args()
    calculate_all_metrics_to_csv(out_csv=args.metrics_csv)

In [ ]:
if __name__ == '__main__':
    main()